### Import Libraries

In [1]:
import os
import boto3
import pickle
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import r2_score
import xgboost as xgb
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline,FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import(
OneHotEncoder,
    MinMaxScaler,
    PowerTransformer,
    FunctionTransformer,
    StandardScaler,
    OrdinalEncoder
)
from sklearn.ensemble import RandomForestRegressor
from feature_engine.selection import SelectBySingleFeaturePerformance
from feature_engine.datetime import DatetimeFeatures
from feature_engine.outliers import Winsorizer
from feature_engine.encoding import (RareLabelEncoder
                                    ,MeanEncoder,CountFrequencyEncoder)

import warnings
import sagemaker
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker.tuner import (
IntegerParameter,
ContinuousParameter,
HyperparameterTuner
)

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "json" in "MonitoringDatasetFormat" shadows an attribute in parent "Base"
  warnings.warn(


[03/02/25 14:06:30] INFO     Found credentials from IAM Role:                                   ]8;id=140498;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=896349;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1132\1132]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [4]:
!pip install feature-engine

  Using cached tzdata-2025.1-py2.py3-none-any.whl.metadata (1.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 93.4 MB/s eta 0:00:00
Using cached tzdata-2025.1-py2.py3-none-any.whl (346 kB)
  Attempting uninstall: pandas
    Found existing installation: pandas 1.5.3
    Uninstalling pandas-1.5.3:
      Successfully uninstalled pandas-1.5.3


### Display Settings

In [2]:
pd.set_option('display.max_columns',None)

In [3]:
sklearn.set_config(transform_output='pandas')

In [4]:
warnings.filterwarnings('ignore')

### Read Datasets

In [11]:
pip install --upgrade pandas

Note: you may need to restart the kernel to use updated packages.


In [5]:
train= pd.read_csv("train.csv")

In [6]:
train

,airline,date_of_journey,source,destination,dep_time,arrival_time,duration,total_stops,additional_info,price
0,Jet Airways,2019-04-01,Kolkata,Banglore,06:30:00,16:20:00,590,1.0,No Info,12996
1,Indigo,2019-06-06,Delhi,Cochin,14:20:00,22:30:00,490,1.0,No Info,6938
2,Indigo,2019-05-06,Kolkata,Banglore,15:15:00,17:45:00,150,0.0,No Info,4804
3,Air India,2019-03-01,Chennai,Kolkata,11:40:00,13:55:00,135,0.0,No Info,19630
4,Indigo,2019-05-21,Banglore,Delhi,21:15:00,00:15:00,180,0.0,No Info,3943
...,...,...,...,...,...,...,...,...,...,...
635,Jet Airways,2019-05-06,Kolkata,Banglore,14:05:00,09:20:00,1155,1.0,In-flight meal not included,9663
636,Jet Airways,2019-05-06,Kolkata,Banglore,17:00:00,23:35:00,395,1.0,No Info,12121
637,Jet Airways,2019-05-12,Kolkata,Banglore,16:30:00,22:35:00,365,1.0,In-flight meal not included,10844
638,Vistara,2019-03-18,Banglore,New Delhi,16:00:00,18:35:00,155,0.0,No Info,11913


In [7]:
val= pd.read_csv('val.csv')
val

,airline,date_of_journey,source,destination,dep_time,arrival_time,duration,total_stops,additional_info,price
0,Multiple Carriers,2019-06-27,Delhi,Cochin,09:15:00,19:00:00,585,1.0,No Info,11622
1,Multiple Carriers,2019-03-06,Delhi,Cochin,09:45:00,16:10:00,385,1.0,No Info,10276
2,Spicejet,2019-05-01,Kolkata,Banglore,17:10:00,19:40:00,150,0.0,No Info,4174
3,Jet Airways,2019-05-01,Banglore,Delhi,11:10:00,14:05:00,175,0.0,No Info,7229
4,Goair,2019-05-03,Banglore,Delhi,20:55:00,23:40:00,165,0.0,No Info,4239
...,...,...,...,...,...,...,...,...,...,...
155,Spicejet,2019-04-24,Kolkata,Banglore,06:55:00,09:30:00,155,0.0,No check-in baggage included,3841
156,Vistara,2019-06-27,Banglore,Delhi,09:45:00,12:35:00,170,0.0,No Info,4668
157,Vistara,2019-06-12,Kolkata,Banglore,07:10:00,10:45:00,1655,1.0,No Info,8452
158,Multiple Carriers,2019-06-09,Delhi,Cochin,11:40:00,01:30:00,830,1.0,No Info,6795


In [8]:
test= pd.read_csv('test.csv')
test

,airline,date_of_journey,source,destination,dep_time,arrival_time,duration,total_stops,additional_info,price
0,Indigo,2019-05-24,Banglore,Delhi,07:10:00,10:05:00,175,0.0,No Info,4823
1,Indigo,2019-03-21,Delhi,Cochin,07:45:00,13:40:00,355,1.0,No Info,6101
2,Jet Airways,2019-05-24,Kolkata,Banglore,20:00:00,23:35:00,1655,1.0,In-flight meal not included,10844
3,Air India,2019-05-21,Kolkata,Banglore,16:50:00,13:45:00,1255,2.0,No Info,13484
4,Air India,2019-06-09,Delhi,Cochin,19:45:00,19:15:00,1410,2.0,No Info,9968
...,...,...,...,...,...,...,...,...,...,...
195,Multiple Carriers,2019-06-27,Delhi,Cochin,13:00:00,19:15:00,375,1.0,No Info,14848
196,Jet Airways,2019-03-24,Kolkata,Banglore,08:25:00,18:10:00,585,1.0,No Info,13759
197,Indigo,2019-06-18,Banglore,Delhi,10:10:00,13:00:00,170,0.0,No Info,3943
198,Multiple Carriers,2019-04-09,Delhi,Cochin,08:45:00,19:15:00,630,1.0,No Info,8371


## Preprocessing Operations

In [9]:
#airline
air_transformer=Pipeline(steps=[
    ("imputer",SimpleImputer(strategy='most_frequent')),
    ("grouper",RareLabelEncoder(tol=0.1,replace_with='Other',n_categories=2)),
    ("encoder",OneHotEncoder(sparse_output=False,handle_unknown='ignore'))
])

#doj
feature_to_extract=["month","year","week","day_of_week","day_of_month","day_of_year"]

doj_transformer=Pipeline(steps=[
    ("dt",DatetimeFeatures(features_to_extract=feature_to_extract,yearfirst=True,format='mixed')),
    ('scaler',MinMaxScaler())
])

#source and destination
location_pipe1=Pipeline(steps=[
    ('grouper',RareLabelEncoder(tol=0.1,replace_with='Other',n_categories=2)),
    ('encoder',MeanEncoder()),
    ('scaler',PowerTransformer())   
])
def is_north(x):
    columns=x.columns.to_list()
    north_cities=['Delhi','Kolkata','New Delhi','Mumbai']
    return (
    x
    .assign(**{
        f"{col}_is_north": x.loc[:,col].isin(north_cities).astype(int)
        for col in columns
    })
    .drop(columns=columns))
location_transformer=FeatureUnion(transformer_list=[
    ("part1",location_pipe1),
    ("part2",FunctionTransformer(func=is_north))
])

# dep_time and arrival_time
time_pipe1=Pipeline(steps=[
    ('dt',DatetimeFeatures(features_to_extract=['hour','minute'])),
    ('scaler',MinMaxScaler())
])
def part_of_day(x,morning=4,noon=12,eve=16,night=20):
    columns=x.columns.to_list()
    x_temp=x.assign(**{
        col: pd.to_datetime(x.loc[:,col]).dt.hour
        for col in columns
    })
    
    return(
    x_temp.assign(**{
        f"{col}_part_of_day":np.select(
        [x_temp.loc[:,col].between(morning,noon,inclusive='left'),
        x_temp.loc[:,col].between(noon,eve,inclusive='left'),
        x_temp.loc[:,col].between(eve,night,inclusive='left')],
        ['morning','afternoon','evening'],default='night')
        
        for col in columns
        
    })
        .drop(columns=columns)
    )
time_pipe2=Pipeline(steps=[
    ('part',FunctionTransformer(func=part_of_day)),
    ('count',CountFrequencyEncoder()),
    ('scaler',MinMaxScaler())
])
time_transformer=FeatureUnion(transformer_list=[
    ("part1",time_pipe1),
    ("part2",time_pipe2)
])

#duration
def duration_of_day(x,short=200,medium=550,long=850):
    columns=x.columns.to_list()
    x_temp=x.assign(**{
        col: pd.to_datetime(x.loc[:,col]).astype('int64') # Convert nanoseconds to microseconds
        for col in columns
    })
    
    return(
    x_temp.assign(**{
        f"{col}_duration_of_day":np.select(
        [x_temp.loc[:,col].between(short,medium,inclusive='left'),
        x_temp.loc[:,col].between(medium,long,inclusive='left')],
        ['short','medium'],default='long')
        
        for col in columns
        
    })
        .drop(columns=columns)
    )
def is_over(X,value=1000):
    return (
    X
    .assign(**{
        f'duration_over_{value}': X.duration.ge(value).astype(int)
    })
    .drop(columns='duration'))

time_transformer=FeatureUnion(transformer_list=[
    ("part1",time_pipe1),
    ("part2",time_pipe2)
])
class RBFPercentileSimilarity(BaseEstimator,TransformerMixin):
    def __init__(self,variables=None,percentiles=[0.25,0.5,0.75],gamma=0.01):
        self.variables=variables
        self.percentiles=percentiles
        self.gamma=gamma
        
        
    def fit(self,X,y=None):
        if not self.variables:
            self.variables=X.select_dtypes(include='number').columns.to_list()
            
        self.reference_values={
            col: (
            X
                .loc[:,col]
                .quantile(self.percentiles)
                .values
                .reshape(-1,1)
            )
            for col in self.variables
        }
        
        return self
    
    def transform(self,X):
        objects=[]
        for col in self.variables:
            columns=[f"{col}_rbf_{int(percentile*100)}"for percentile in self.percentiles]
            obj=pd.DataFrame(
            data=rbf_kernel(X.loc[:,[col]],Y=self.reference_values[col],gamma=self.gamma),columns=columns)
            objects.append(obj)
            return pd.concat(objects,axis=1)
duration_pipe1=Pipeline(steps=[
    ('rbf',RBFPercentileSimilarity()),
    ('scaler',PowerTransformer())
])

duration_pipe2=Pipeline(steps=[
    ('cat',FunctionTransformer(func=duration_of_day)),
    ('encoder',OrdinalEncoder(categories=[['short','medium','long']]))
])
duration_union=FeatureUnion(transformer_list=[
    ('part1',duration_pipe1),
    ('part2',duration_pipe2),
    ('part3',FunctionTransformer(func=is_over)),
    ('part4', StandardScaler())
])
duration_transformer=Pipeline(steps=[
    ('outliers',Winsorizer(capping_method='iqr',fold=1.5)),
    ('imputer',SimpleImputer(strategy='median')),
    ('union',duration_union)
])

#total_stops
def is_direct(X):
    return X.assign(is_direct_flight=X.total_stops.eq(0).astype(int))


total_stops_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("", FunctionTransformer(func=is_direct))
])

#additional_info
info_pipe1 = Pipeline(steps=[
    ("group", RareLabelEncoder(tol=0.1, n_categories=2, replace_with="Other")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])
def have_info(X):
    return X.assign(additional_info=X.additional_info.ne("No Info").astype(int))
info_union = FeatureUnion(transformer_list=[
    ("part1", info_pipe1),
    ("part2", FunctionTransformer(func=have_info))
])
info_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
    ("union", info_union)
])

# column transformer
column_transformer=ColumnTransformer(transformers=[
    ("air",air_transformer,['airline']),
    ("doj",doj_transformer,['date_of_journey']),
    ("loc",location_transformer,['source','destination']),
    ("time",time_transformer,['dep_time','arrival_time']),
    ("duration",duration_transformer,['duration']),
    ("total_stops",total_stops_transformer,['total_stops']),
    ("info", info_transformer, ["additional_info"])
])

#feature selection
estimator = RandomForestRegressor(n_estimators=10, max_depth=3, random_state=42)

selector = SelectBySingleFeaturePerformance(
    estimator=estimator,
    scoring="r2",
    threshold=0.1 #threshold for r2 score
)
preprocessor = Pipeline(steps=[
    ("ct", column_transformer),
    ("selector", selector)
])


In [10]:
preprocessor.fit(
train.drop(columns='price'),
train.price.copy())

Pipeline(steps=[('ct',
                 ColumnTransformer(transformers=[('air',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('grouper',
                                                                   RareLabelEncoder(n_categories=2,
                                                                                    replace_with='Other',
                                                                                    tol=0.1)),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['airline']),
                                                 ('doj',
                                                  Pipeline(steps=[('dt',
                                                                   DatetimeFeatures(features_to_ext...
                                                                                                                                     replace_with='Other',
                                                                                                                                     tol=0.1)),
                                                                                                                   ('encoder',
                                                                                                                    OneHotEncoder(handle_unknown='ignore',
                                                                                                                                  sparse_output=False))])),
                                                                                                  ('part2',
                                                                                                   FunctionTransformer(func=<function have_info at 0x7f02ef7b4a60>))]))]),
                                                  ['additional_info'])])),
                ('selector',
                 SelectBySingleFeaturePerformance(estimator=RandomForestRegressor(max_depth=3,
                                                                                  n_estimators=10,
                                                                                  random_state=42),
                                                  scoring='r2',
                                                  threshold=0.1))])

In [13]:
preprocessor.transform(train.drop(columns='price'))

,air__airline_Indigo,air__airline_Jet Airways,doj__date_of_journey_week,doj__date_of_journey_day_of_year,loc__destination,duration__duration_rbf_25,duration__duration,total_stops__total_stops,total_stops__is_direct_flight
0,0.0,1.0,0.294118,0.262712,-0.455670,-0.449168,-0.092029,1.0,0
1,1.0,0.0,0.823529,0.822034,1.073094,-0.449168,-0.298114,1.0,0
2,1.0,0.0,0.588235,0.559322,-0.455670,-0.321266,-0.998800,0.0,1
3,0.0,0.0,0.000000,0.000000,-0.578469,-0.449157,-1.029712,0.0,1
4,1.0,0.0,0.705882,0.686441,-1.904911,2.484770,-0.936974,0.0,1
...,...,...,...,...,...,...,...,...,...
635,0.0,1.0,0.588235,0.559322,-0.455670,-0.449168,1.072346,1.0,0
636,0.0,1.0,0.588235,0.559322,-0.455670,-0.449168,-0.493894,1.0,0
637,0.0,1.0,0.588235,0.610169,-0.455670,-0.449168,-0.555719,1.0,0
638,0.0,0.0,0.176471,0.144068,-0.578469,0.455201,-0.988495,0.0,1


### Preprocess Data and Upload to Bucket

In [14]:
bucket_name= 'sagemaker-flights-price'
data_prefix='data'

In [15]:
def get_file_name(name):
    return f'{name}-pre.csv'

In [16]:
def export_data(data,name,pre):
    #split data into X and y subsets
    X= data.drop(columns='price')
    y=data.price.copy()
    
    #transformation
    X_pre=pre.transform(X)
    
    #exporting
    file_name=get_file_name(name)
    y.to_frame().join(X_pre).to_csv(file_name,index=False,header=False)

In [17]:
def upload_to_bucket(name):
    file_name=get_file_name(name)
    boto3.Session().resource('s3').Bucket(bucket_name).Object(os.path.join(data_prefix,f"{name}/{name}.csv")).upload_file(file_name)

In [18]:
def export_and_upload_bucket(data,name,pre):
    export_data(data,name,pre)
    upload_to_bucket(name)

In [19]:
export_and_upload_bucket(train,'train',preprocessor)

[03/02/25 14:09:32] INFO     Found credentials from IAM Role:                                   ]8;id=897861;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=271281;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1132\1132]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

In [20]:
export_and_upload_bucket(val,'val',preprocessor)

[03/02/25 14:09:35] INFO     Found credentials from IAM Role:                                   ]8;id=493528;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=42382;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1132\1132]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

In [21]:
export_and_upload_bucket(test,'test',preprocessor)

[03/02/25 14:09:37] INFO     Found credentials from IAM Role:                                   ]8;id=878690;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=765221;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1132\1132]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

### Model and Hyperparameter Tuning

In [22]:
session=sagemaker.Session()
region_name= session.boto_region_name

[03/02/25 14:09:42] INFO     Found credentials from IAM Role:                                   ]8;id=5881;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=938624;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1132\1132]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

In [23]:
output_path=f's3://{bucket_name}/model/output'

In [24]:
model=Estimator(
image_uri=sagemaker.image_uris.retrieve('xgboost',region_name,'1.7-1'),
role=sagemaker.get_execution_role(),
instance_count=1,
instance_type='ml.m4.xlarge',
volume_size=5,
output_path=output_path,
use_spot_instances=True,
max_run=300,
max_wait=700,
sagemaker_session=session)

[03/02/25 14:11:46] INFO     Ignoring unnecessary instance type: None.                            ]8;id=288255;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/image_uris.py\image_uris.py]8;;\:]8;id=416377;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/image_uris.py#530\530]8;;\

                    INFO     Found credentials from IAM Role:                                   ]8;id=491679;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=410830;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1132\1132]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

                    INFO     Found credentials from IAM Role:                                   ]8;id=797625;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=319964;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1132\1132]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

In [25]:
model.set_hyperparameters(
    objective="reg:linear",
    num_round=10,
    eta=0.1,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    alpha=0.1
)

In [26]:
hyperparameter_ranges = {
    "eta": ContinuousParameter(0.05, 0.3),
    "alpha": ContinuousParameter(0, 1),
    "max_depth": IntegerParameter(3, 6)
}

In [27]:
tuner = HyperparameterTuner(
    estimator=model,
    objective_metric_name="validation:rmse",
    hyperparameter_ranges=hyperparameter_ranges,
    strategy="Bayesian",
    objective_type="Minimize"
)

### Data Channels

In [28]:
def get_data_channel(name):
    bucket_path = f"s3://{bucket_name}/{data_prefix}/{name}"
    return TrainingInput(bucket_path, content_type="csv")

In [29]:
train_data_channel = get_data_channel("train")
train_data_channel

In [30]:
val_data_channel = get_data_channel("val")
val_data_channel

In [31]:
data_channels = {
    "train": train_data_channel,
    "validation": val_data_channel
}

### Train and Tune the Model

In [32]:
tuner.fit(data_channels)

[03/02/25 14:13:10] WARNING  No finished training job found associated with this estimator.       ]8;id=46610;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/estimator.py\estimator.py]8;;\:]8;id=244156;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/estimator.py#1914\1914]8;;\
                             Please make sure this estimator is only used for building workflow                    
                             config                                                                                

                    WARNING  No finished training job found associated with this estimator.       ]8;id=308448;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/estimator.py\estimator.py]8;;\:]8;id=267891;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/estimator.py#1914\1914]8;;\
                             Please make sure this estimator is only used for building workflow                    
                             config                                                                                

                    INFO     Creating hyperparameter tuning job with name:                          ]8;id=325366;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=251405;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py#3383\3383]8;;\
                             sagemaker-xgboost-250302-1413                                                         

.............................................!


In [ ]:
#tuner.best_estimator().deploy() with this we can deploy the model in sagemaker

### Model Evaluation

In [33]:
with open('xgboost-model','rb') as f:
    best_model=pickle.load(f)
    
best_model

In [54]:
def evaluate_model(file_name):
    data=pd.read_csv(file_name)
    
    X=xgb.DMatrix(data.iloc[:,1:])
    y= data.iloc[:,0].copy()
    
    pred=best_model.predict(X)
    return r2_score(y,pred)

In [55]:
evaluate_model('train-pre.csv')

0.4228367209434509

In [56]:
evaluate_model('val-pre.csv')

0.4010060429573059

In [57]:
evaluate_model('test-pre.csv')

0.3798335790634155